# NVIDIA NIM Blueprints

[NVIDIA NIM Blueprints](https://build.nvidia.com/blueprints) are a collection of pre-trained, customizable reference AI workflows designed to streamline the development and deployment of generative AI applications across a variety of industries. 

<div align="center">
    <img src="imgs/blueprints_cover_page.png" style="width: 1200px; height: 600px;" />
</div>

This notebook covers a sample deployment of a NIM Blueprint for [Enterprise Grade RAG](https://build.nvidia.com/nvidia/build-an-enterprise-rag-pipeline) application.

<div align="center">
    <img src="imgs/rag_bp.jpg" style="width: 1000px; height: 800px;" />
</div>


These pre-defined reference workflows give a strong starting point with key features:
* Multimodal PDF data extraction support with text, tables, charts, and infographics
* Hybrid search with dense and sparse search
* Opt-in image captioning with vision language models (VLMs)
* Reranking to further improve accuracy
* GPU-accelerated Index creation and search
* Multi-turn conversations
* Multi-session support
* Telemetry and observability
* Opt-in for reflection to improve accuracy
* Opt-in for guardrailing conversations
* Sample user interface
* OpenAI-compatible APIs
* Decomposable and customizable

The codebases for the various NIM Blueprints can be found at [Github](https://github.com/NVIDIA-AI-Blueprints/)

Rest of the commands are run in a **terminal window**. Open a new terminal window and start running the code below:

## Pull the required code

```
git clone https://github.com/NVIDIA-AI-Blueprints/rag.git
```

To get started, this deployment will use the NVIDIA NIM Cloud endpoint for accessing the LLMs. Other containers required for monitoring, telemetry and Vector Database would be hosted on the machine.

## Setup the required environment variables

```
export NVIDIA_API_KEY="<PUT_YOUR_API_KEY_HERE>"
export APP_EMBEDDINGS_SERVERURL=""
export APP_LLM_SERVERURL=""
export APP_RANKING_SERVERURL=""
export EMBEDDING_NIM_ENDPOINT="https://integrate.api.nvidia.com/v1"
export PADDLE_HTTP_ENDPOINT="https://ai.api.nvidia.com/v1/cv/baidu/paddleocr"
export PADDLE_INFER_PROTOCOL="http"
export YOLOX_HTTP_ENDPOINT="https://ai.api.nvidia.com/v1/cv/nvidia/nemoretriever-page-elements-v2"
export YOLOX_INFER_PROTOCOL="http"
export YOLOX_GRAPHIC_ELEMENTS_HTTP_ENDPOINT="https://ai.api.nvidia.com/v1/cv/nvidia/nemoretriever-graphic-elements-v1"
export YOLOX_GRAPHIC_ELEMENTS_INFER_PROTOCOL="http"
export YOLOX_TABLE_STRUCTURE_HTTP_ENDPOINT="https://ai.api.nvidia.com/v1/cv/nvidia/nemoretriever-table-structure-v1"
export YOLOX_TABLE_STRUCTURE_INFER_PROTOCOL="http"
```

Start the vector db containers from the repo root.
   ```bash
   docker compose -f rag/deploy/compose/vectordb.yaml up -d
   ```

Start the ingestion containers from the repo root. This pulls the prebuilt containers from NGC and deploys it on your system.

   ```bash
   docker compose -f rag/deploy/compose/docker-compose-ingestor-server.yaml up -d
   ```

Start the rag containers from the repo root. This pulls the prebuilt containers from NGC and deploys it on your system.

   ```bash
   docker compose -f rag/deploy/compose/docker-compose-rag-server.yaml up -d
   ```


Confirm all the below mentioned containers are running.

   ```bash
   docker ps --format "table {{.ID}}\t{{.Names}}\t{{.Status}}"
   ```

   *Example Output*

   ```output
   NAMES                                   STATUS
   compose-nv-ingest-ms-runtime-1          Up 5 minutes (healthy)
   ingestor-server                         Up 5 minutes
   compose-redis-1                         Up 5 minutes
   rag-playground                          Up 9 minutes
   rag-server                              Up 9 minutes
   milvus-standalone                       Up 36 minutes
   milvus-minio                            Up 35 minutes (healthy)
   milvus-etcd                             Up 35 minutes (healthy)
   ```

Open a web browser and access `http://localhost:8090` to use the RAG Playground. 

PS: If you are in a cluster environment, please ensure you do a port forwarding of a port to port 8090 to allow web traffic through your browser.

## Output

The blueprint gives access to a RAG-playground.

<img src="imgs/rag_blueprint_output.png" style="width: 1000px; height: 600PX;" >

The blueprint also allows for the flexibility to use the backend API services. The details of the same can be found at [here](https://github.com/NVIDIA-AI-Blueprints/rag/tree/v2.0.0/notebooks)

## Observability
The blueprint supports tracing and observability for the RAG Server using OpenTelemetry (OTel) Collector and Zipkin.

The observability stack consists of:

* OTel Collector - Collects, processes, and exports telemetry data.
* Zipkin - Used for visualising traces.



Before starting the observability services, set the required environment variable for the OTel Collector Config:

In the terminal window:

```
export OPENTELEMETRY_CONFIG_FILE=$(pwd)/rag/deploy/config/otel-collector-config.yaml
```

Run the following command to start the OTel Collector and Zipkin:
```
docker-compose -f deploy/compose/observability.yaml up -d
```


The **RAG Server** needs to have tracing enabled. To do this:
- Ensure that the environment variable `APP_TRACING_ENABLED` is set to `"True"` in `docker-compose-rag-server.yaml`:

```yaml
services:
  rag-server:
    environment:
      # Tracing
      APP_TRACING_ENABLED: "True"
```


## Exercise:

Check out the various Docker config files for Docker deployment under:
* `rag/deploy/compose`
* `rag/src/`

and make amends for the LLMs to use locally deployed NVIDIA NIM and repeat the process.

---
## Licensing

Copyright © 2025 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.